# High-pressure Co–Bi DFT verification (50 GPa)

Three phases reported on the convex hull at 50 GPa:

| Composition | Space group | Prototype |
|-------------|-------------|-----------|
| CoBi₃      | Cmcm (#63)  | PuBr₃    |
| CoBi₂      | I4/mcm (#140) | CuAl₂ |
| CoBi        | Cmcm (#63)  | TlI       |

This notebook builds the structures from space-group + Wyckoff positions,
visualises each unit cell, generates QE `pw.x` inputs at **50 GPa** (`press = 500 kbar`),
runs the calculations, then parses and scores formation energies.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from pymatgen.core import Composition, Lattice, Structure

REPO = Path.cwd().resolve()
if not (REPO / "data").is_dir() and (REPO.parent / "data").is_dir():
    REPO = REPO.parent
print("REPO =", REPO)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

## 1. Build structures from space-group prototypes

In [ ]:
# CoBi3 — PuBr3-type, Cmcm (#63), orthorhombic
# Wyckoff: Co at 4c (0, y, 1/4), Bi1 at 4c (0, y, 1/4), Bi2 at 8f (0, y, z)
cobi3 = Structure.from_spacegroup(
    "Cmcm",
    Lattice.orthorhombic(4.30, 11.50, 8.50),
    ["Co", "Bi", "Bi"],
    [[0.0, 0.250, 0.25],
     [0.0, 0.580, 0.25],
     [0.0, 0.350, 0.060]],
)
print(f"CoBi3 (PuBr3): {cobi3.composition.reduced_formula}, "
      f"{len(cobi3)} atoms, SG Cmcm")
print(cobi3.lattice)
print()

# CoBi2 — CuAl2-type, I4/mcm (#140), tetragonal
# Wyckoff: Co at 4a (0, 0, 1/4), Bi at 8h (x, x+1/2, 0)
cobi2 = Structure.from_spacegroup(
    "I4/mcm",
    Lattice.tetragonal(6.50, 5.30),
    ["Co", "Bi"],
    [[0.0, 0.0, 0.25],
     [0.158, 0.658, 0.0]],
)
print(f"CoBi2 (CuAl2): {cobi2.composition.reduced_formula}, "
      f"{len(cobi2)} atoms, SG I4/mcm")
print(cobi2.lattice)
print()

# CoBi — TlI-type, Cmcm (#63), orthorhombic
# Wyckoff: Co at 4c (0, y, 1/4), Bi at 4c (0, y, 1/4)
cobi = Structure.from_spacegroup(
    "Cmcm",
    Lattice.orthorhombic(4.20, 11.00, 4.70),
    ["Co", "Bi"],
    [[0.0, 0.140, 0.25],
     [0.0, 0.430, 0.25]],
)
print(f"CoBi (TlI): {cobi.composition.reduced_formula}, "
      f"{len(cobi)} atoms, SG Cmcm")
print(cobi.lattice)

structures = {
    "CoBi3_PuBr3": cobi3,
    "CoBi2_CuAl2": cobi2,
    "CoBi_TlI": cobi,
}

## 2. Visualise unit cells

In [ ]:
COLORS = {"Co": "#3366cc", "Bi": "#cc6633"}
SIZES = {"Co": 100, "Bi": 160}


def _unit_cell_edges(lattice):
    """Return line segments for the 12 edges of a parallelepiped."""
    o = np.zeros(3)
    a, b, c = lattice.matrix
    corners = [o, a, b, c, a + b, a + c, b + c, a + b + c]
    edges = [
        (0, 1), (0, 2), (0, 3), (1, 4), (1, 5), (2, 4),
        (2, 6), (3, 5), (3, 6), (4, 7), (5, 7), (6, 7),
    ]
    return [[corners[i], corners[j]] for i, j in edges]


fig, axes = plt.subplots(1, 3, figsize=(18, 5.5),
                         subplot_kw={"projection": "3d"})

for ax, (label, struct) in zip(axes, structures.items()):
    cart = struct.cart_coords
    species = [str(s.specie) for s in struct]

    for sym in set(species):
        mask = np.array([s == sym for s in species])
        ax.scatter(
            cart[mask, 0], cart[mask, 1], cart[mask, 2],
            s=SIZES.get(sym, 100), c=COLORS.get(sym, "gray"),
            label=sym, edgecolors="k", linewidths=0.5, depthshade=True,
        )

    edges = _unit_cell_edges(struct.lattice)
    ax.add_collection3d(Line3DCollection(edges, colors="gray", linewidths=0.8, linestyles="--"))

    all_pts = np.array([p for seg in edges for p in seg])
    for i, dim in enumerate(["x", "y", "z"]):
        lo, hi = all_pts[:, i].min() - 0.5, all_pts[:, i].max() + 0.5
        getattr(ax, f"set_{dim}lim")(lo, hi)

    comp = struct.composition.reduced_formula
    proto = label.split("_")[1]
    ax.set_title(f"{comp} ({proto}-type)\n{len(struct)} atoms/cell", fontsize=11)
    ax.set_xlabel("x (Å)")
    ax.set_ylabel("y (Å)")
    ax.set_zlabel("z (Å)")
    ax.legend(fontsize=8, loc="upper left")

fig.suptitle("Co–Bi phases at 50 GPa (initial unit cells before relaxation)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 3. Generate QE inputs at 50 GPa

In [ ]:
from hullgap.dft.make_qe_inputs import coarse_relax_pw_input, PSEUDO_DIR_DEFAULT

PRESS_KBAR = 500.0   # 50 GPa
RUN_ROOT = REPO / "dft" / "runs" / "Co-Bi_50GPa"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

for label, struct in structures.items():
    run_dir = RUN_ROOT / label
    run_dir.mkdir(parents=True, exist_ok=True)

    pw = coarse_relax_pw_input(
        struct,
        pseudo_dir=PSEUDO_DIR_DEFAULT,
        ecutwfc=40.0,
        ecutrho=320.0,
        kppa=300,
        press_kbar=PRESS_KBAR,
    )
    pw.write_file(str(run_dir / "pw.in"))

    struct.to(filename=str(run_dir / "initial_structure.cif"))
    print(f"  {label}: {run_dir / 'pw.in'}")

print(f"\nAll inputs under {RUN_ROOT}")

## 4. Run QE `pw.x` for all three phases

Each vc-relax at 50 GPa runs via `mpirun`. This may take a while on a local machine.

In [ ]:
import os, time

env = os.environ.copy()
env["PATH"] = "/home/hhoechter/software/qe-7.5/bin:" + env.get("PATH", "")

NP = 4  # MPI ranks

procs = {}
for label in structures:
    run_dir = RUN_ROOT / label
    pw_out = run_dir / "pw.out"
    with open(pw_out, "w") as fout:
        p = subprocess.Popen(
            ["mpirun", "--oversubscribe", "-np", str(NP), "pw.x", "-in", "pw.in"],
            cwd=str(run_dir), stdout=fout, stderr=subprocess.STDOUT, env=env,
        )
    procs[label] = p
    print(f"  Started {label}  (PID {p.pid})")

print(f"\n{len(procs)} jobs launched. Waiting for completion...")

results = {}
while procs:
    for label in list(procs):
        rc = procs[label].poll()
        if rc is not None:
            results[label] = rc
            print(f"  {label} finished  (exit {rc})")
            del procs[label]
    if procs:
        time.sleep(30)

print("\nAll done.", results)

## 5. Parse QE outputs

In [ ]:
from hullgap.dft.parse_qe_outputs import parse_run_tree

parsed = parse_run_tree(RUN_ROOT)
parsed

## 6. Formation energies and convex hull at 50 GPa

**Note:** The elemental reference energies below are from our 0 GPa SCF calculations.
For a rigorous 50 GPa hull the references should be re-run at 50 GPa too.
For hackathon ranking purposes the *relative* ordering is still meaningful.

In [ ]:
from hullgap.dft.dft_hull import (
    formation_energy_per_atom,
    hull_energy_at_x,
    load_elemental_references,
    lower_convex_hull_2d,
    score_dft_candidates,
    x_bi_fraction,
)

refs = load_elemental_references(REPO / "dft" / "reference_energies.yaml")
scored = score_dft_candidates(parsed, "Co-Bi", refs)
scored

In [ ]:
hull_pts = [(0.0, 0.0), (1.0, 0.0)]
for _, r in scored.iterrows():
    if r.get("validation_status") == "failed_dft":
        continue
    x = float(r["x_Bi"])
    e = float(r["formation_energy_eV_atom"])
    hull_pts.append((x, e))

hull_arr = lower_convex_hull_2d(np.array(hull_pts, dtype=float))
xs = np.linspace(0, 1, 200)
ys = np.array([hull_energy_at_x(hull_arr, x) for x in xs])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(xs, ys, color="#333", lw=2, label="Lower hull")

markers = {"CoBi3": "s", "CoBi2": "D", "CoBi": "o"}
for _, r in scored.iterrows():
    if r.get("validation_status") == "failed_dft":
        continue
    comp = Composition(str(r["formula"]))
    lab = comp.reduced_formula
    x = float(r["x_Bi"])
    e = float(r["formation_energy_eV_atom"])
    m = markers.get(lab, "o")
    ax.scatter([x], [e], s=100, zorder=5, marker=m, label=f"{lab}  ({e:+.3f} eV/at)")

ax.axhline(0.0, color="gray", ls="--", lw=1)
ax.set_xlabel(r"Mole fraction Bi ($x_{\mathrm{Bi}}$)")
ax.set_ylabel("Formation energy (eV / atom)")
ax.set_title("Co\u2013Bi convex hull at 50 GPa (QE PBE/PAW vc-relax)")
ax.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()